In [3]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from utils import *
from interpolation import *


In [2]:
files = sorted(glob.glob('/home/ulyanov/data/cross/*'))
files

['/home/ulyanov/data/cross/hmi.M_45s.20241120_001500_TAI.2.magnetogram.fits',
 '/home/ulyanov/data/cross/solo_L1_phi-fdt-ilam_20241120T001503_V202412020209C_0451200501.fits.gz',
 '/home/ulyanov/data/cross/solo_L2_phi-fdt-blos_20241120T001503_V202602220151_0451200501.fits.gz']

In [3]:
s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [19]:
file_hmi = files[0]
file_fdt = files[2]

with fits.open(file_hmi) as hdul:
    header_hmi = hdul[1].header
    data_hmi = hdul[1].data

with fits.open(file_fdt) as hdul:
    header_fdt = hdul[0].header
    data_fdt = hdul[0].data

data_fdt = undistort(data_fdt, header_fdt, xd, yd)

data_hmi = rebin(data_hmi, 8, update_header=header_hmi)
data_hmi = reproject(data_hmi, header_hmi, header_fdt, correct_mu=True, mu_thr=0.5, crota=header_fdt['CROTA'] + 0.25, rsun=330.25 + 0.2, xc=421.64, yc=457.87)

data_fdt[np.isnan(data_hmi)] = np.nan

In [20]:
xx, yy = [0], [0]

for q in np.arange(2., 30, 0.5):
    t = np.where(np.sqrt(data_hmi ** 2 + data_fdt ** 2) < q ** 2)
    x, y = data_hmi[t], data_fdt[t]

    A = np.array([[np.mean(x ** 2), np.mean(x * y)],
                  [np.mean(x * y), np.mean(y ** 2)]])

    vals, vecs = np.linalg.eigh(A)
    u, v = vecs[:, 1]
    u, v = u * q ** 2 * np.sign(u), v * q ** 2 * np.sign(u)

    xx = [-u] + xx + [u]
    yy = [-v] + yy + [v]

xx = np.array(xx)
yy = np.array(yy)

In [21]:
plt.figure(figsize=(10,10))
plt.scatter(data_hmi, data_fdt, s=0.01)
plt.plot(xx, yy, '--', color='black')

#plt.xlim(-40,40)
#plt.ylim(-40,40)

plt.xlim(-20,20)
plt.ylim(-20,20)

(-20.0, 20.0)

In [27]:
np.savez('cross_calibration.npz', hmi=xx, fdt=yy)

In [22]:
data_fdt_ = hmize(data_fdt, yy, xx)

In [24]:
plt.figure(figsize=(10,10))
plt.scatter(data_hmi, data_fdt_, s=0.01)

plt.xlim(-500,500)
plt.ylim(-500,500)

(-500.0, 500.0)

In [25]:
plt.figure(figsize=(10,10))
plt.imshow(data_hmi, cmap='seismic', vmin=-50, vmax=50)
plt.tight_layout()

In [26]:
plt.figure(figsize=(10,10))
plt.imshow(data_fdt_, cmap='seismic', vmin=-50, vmax=50)
plt.tight_layout()

In [1]:
10 * 12 * 3600

432000

In [4]:
0.5 / 700 * 180 / np.pi

0.0409255567950588